# Fake or Real News with NLP

<img src='https://childrens-binary.files.bbci.co.uk/childrens-binarystore/cbbc/I-Newspaper2.jpg'>

Bu çalışmanın amacı, metin verileri üzerinden haberlerin doğruluğunu otomatik olarak sınıflandırabilen bir model geliştirmek ve farklı makine öğrenmesi ile derin öğrenme yöntemlerinin performanslarını karşılaştırmaktır.

The aim of this study is to develop a model that can automatically classify the accuracy of news articles from text data and to compare the performance of different machine learning and deep learning methods.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

import warnings 
warnings.filterwarnings('ignore')

C:\Users\LENOVO\anaconda3\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
df=pd.read_csv('fake_or_real_news.csv')

In [3]:
df.head()

,Unnamed: 0,title,text,label
0,8476,You Can Smell Hillary’s Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE
1,10294,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE
2,3608,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL
3,10142,Bernie supporters on Twitter erupt in anger ag...,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",FAKE
4,875,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,REAL


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6335 entries, 0 to 6334
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  6335 non-null   int64 
 1   title       6335 non-null   object
 2   text        6335 non-null   object
 3   label       6335 non-null   object
dtypes: int64(1), object(3)
memory usage: 198.1+ KB


In [5]:
df.isnull().sum()

Unnamed: 0    0
title         0
text          0
label         0
dtype: int64

In [6]:
df.label.value_counts()

label
REAL    3171
FAKE    3164
Name: count, dtype: int64

In [7]:
#Feature Enginerring

In [8]:
df['label']=df['label'].replace({'REAL':1,'FAKE':0})

In [9]:
df['text']=df['title']+" "+df['text']

In [10]:
df.head()

,Unnamed: 0,title,text,label
0,8476,You Can Smell Hillary’s Fear,You Can Smell Hillary’s Fear Daniel Greenfield...,0
1,10294,Watch The Exact Moment Paul Ryan Committed Pol...,Watch The Exact Moment Paul Ryan Committed Pol...,0
2,3608,Kerry to go to Paris in gesture of sympathy,Kerry to go to Paris in gesture of sympathy U....,1
3,10142,Bernie supporters on Twitter erupt in anger ag...,Bernie supporters on Twitter erupt in anger ag...,0
4,875,The Battle of New York: Why This Primary Matters,The Battle of New York: Why This Primary Matte...,1


In [11]:
df = df.drop(columns=["Unnamed: 0", "title"])

In [12]:
df['text']=df['text'].str.lower() # küçük harfe çeviriyor
df['text']=df['text'].str.replace('[^\w\s]','', regex=True) # noktolama işaretlerini kaldırır
df['text']=df['text'].str.replace('\d+','', regex=True) # rakamları kaldırır
df['text']=df['text'].str.replace('\n','', regex=True) # satır sonlarını kaldırır
df['text']=df['text'].str.replace('\r','', regex=True) # enter'ları kaldırıyor

In [13]:
x=df['text']
y=df['label']

In [14]:
x.head()

0    you can smell hillarys fear daniel greenfield ...
1    watch the exact moment paul ryan committed pol...
2    kerry to go to paris in gesture of sympathy us...
3    bernie supporters on twitter erupt in anger ag...
4    the battle of new york why this primary matter...
Name: text, dtype: object

In [15]:
y.head()

0    0
1    0
2    1
3    0
4    1
Name: label, dtype: int64

In [16]:
from textblob import TextBlob
import nltk
from nltk.corpus import stopwords

# NLTK durak kelimeleri indir
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\LENOVO\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [17]:
#tokenizasyon ve kök bulma (lematizasyon) 
def ekkok(text):
    words = TextBlob(text).words
    return [word.lemmatize() for word in words if word.lower() not in stop_words]

In [18]:
vect=CountVectorizer()

In [19]:
vect = CountVectorizer(ngram_range=(1, 2),max_features=20000, min_df=5, analyzer=ekkok, stop_words='english' )

In [20]:
x_vect = vect.fit_transform(x).toarray()

In [21]:
x_train,x_test,y_train,y_test=train_test_split(x_vect,y,test_size=.2,random_state=42)

In [22]:
tf = pd.DataFrame(vect.transform(x).toarray(), columns=vect.get_feature_names_out())

In [23]:
tf

,aa,aap,aaron,ab,abaaoud,abaaouds,aback,abadi,abandon,abandoned,...,zip,zombie,zone,zoning,zoo,zucker,zuckerberg,zuesse,½,â
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6330,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6331,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
6332,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6333,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [24]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

model = Sequential([
    Dense(128, activation='relu', input_shape=(x_vect.shape[1],)),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')  
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',  
    metrics=['accuracy']
)

In [25]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [26]:
model.fit(x_train, y_train, batch_size=32, validation_data=(x_test, y_test), epochs=15, verbose=2, callbacks=[early_stop])

Epoch 1/15
159/159 - 9s - 60ms/step - accuracy: 0.8650 - loss: 0.3463 - val_accuracy: 0.9234 - val_loss: 0.2352
Epoch 2/15
159/159 - 6s - 40ms/step - accuracy: 0.9603 - loss: 0.1390 - val_accuracy: 0.9274 - val_loss: 0.2388
Epoch 3/15
159/159 - 6s - 40ms/step - accuracy: 0.9834 - loss: 0.0642 - val_accuracy: 0.9250 - val_loss: 0.2679
Epoch 4/15
159/159 - 6s - 39ms/step - accuracy: 0.9923 - loss: 0.0437 - val_accuracy: 0.9321 - val_loss: 0.3114
Epoch 5/15
159/159 - 6s - 39ms/step - accuracy: 0.9909 - loss: 0.0400 - val_accuracy: 0.9369 - val_loss: 0.2957
Epoch 6/15
159/159 - 7s - 41ms/step - accuracy: 0.9964 - loss: 0.0127 - val_accuracy: 0.9345 - val_loss: 0.3781
Epoch 7/15
159/159 - 6s - 38ms/step - accuracy: 0.9966 - loss: 0.0137 - val_accuracy: 0.9353 - val_loss: 0.3412
Epoch 8/15
159/159 - 6s - 39ms/step - accuracy: 0.9964 - loss: 0.0136 - val_accuracy: 0.9369 - val_loss: 0.4181
Epoch 9/15
159/159 - 6s - 40ms/step - accuracy: 0.9931 - loss: 0.0386 - val_accuracy: 0.9250 - val_loss:

In [27]:
model.evaluate(x_test, y_test)

40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9234 - loss: 0.2352


[0.2351643145084381, 0.9234411716461182]

In [28]:
model.save('fakeorreal_model.h5')

In [29]:
model.save('skin_cancer_model.keras')

In [30]:
fake = df[df['label']==0]['text'].iloc[0]
real = df[df['label']==1]['text'].iloc[0]

In [31]:
fake

'you can smell hillarys fear daniel greenfield a shillman journalism fellow at the freedom center is a new york writer focusing on radical islam in the final stretch of the election hillary rodham clinton has gone to war with the fbi the word unprecedented has been thrown around so often this election that it ought to be retired but its still unprecedented for the nominee of a major political party to go war with the fbi but thats exactly what hillary and her people have done coma patients just waking up now and watching an hour of cnn from their hospital beds would assume that fbi director james comey is hillarys opponent in this election the fbi is under attack by everyone from obama to cnn hillarys people have circulated a letter attacking comey there are currently more media hit pieces lambasting him than targeting trump it wouldnt be too surprising if the clintons or their allies were to start running attack ads against the fbi the fbis leadership is being warned that the entire l

In [32]:
real

'kerry to go to paris in gesture of sympathy us secretary of state john f kerry said monday that he will stop in paris later this week amid criticism that no top american officials attended sundays unity march against terrorismkerry said he expects to arrive in paris thursday evening as he heads home after a week abroad he said he will fly to france at the conclusion of a series of meetings scheduled for thursday in sofia bulgaria he plans to meet the next day with foreign minister laurent fabius and president francois hollande then return to washingtonthe visit by kerry who has family and childhood ties to the country and speaks fluent french could address some of the criticism that the united states snubbed france in its darkest hour in many yearsthe french press on monday was filled with questions about why neither president obama nor kerry attended sundays march as about  leaders of other nations did obama was said to have stayed away because his own security needs can be taxing on

In [33]:
text1=vect.transform([fake]).toarray()
p1 = model.predict(text1)
label1 = (p1 > 0.5).astype(int).flatten()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 183ms/step


In [34]:
label1

array([0])

In [35]:
text2=vect.transform([real]).toarray()
p2= model.predict(text2)
label2 = (p2 > 0.5).astype(int).flatten()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step


In [36]:
label2

array([1])

In [37]:
# SVC MODELİ SVC, Support Vector Machine (SVM) algoritmasının sınıflandırma (classification) versiyonudur.
#from sklearn.svm import SVC
#s=SVC()

In [38]:
#model=s.fit(x_train,y_train)
#tahmin=model.predict(x_test)

In [39]:
#accuracy_score(tahmin,y_test) 0.8784530386740331

In [40]:
# Machine Learning Algoritms

In [41]:
ybw=df[(df.label==0) | (df.label==1) ] 

In [42]:
ybw.reset_index(drop=True, inplace=True)

In [43]:
x=ybw[['text']]
y=ybw[['label']]

In [44]:
x.head()

,text
0,you can smell hillarys fear daniel greenfield ...
1,watch the exact moment paul ryan committed pol...
2,kerry to go to paris in gesture of sympathy us...
3,bernie supporters on twitter erupt in anger ag...
4,the battle of new york why this primary matter...


In [45]:
y.head()

,label
0,0
1,0
2,1
3,0
4,1


In [46]:
yenix=vect.fit_transform(x['text'])

In [47]:
x_train, x_test, y_train, y_test=train_test_split(yenix,y, random_state=42, test_size=.20)

In [48]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import BernoulliNB

from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

b = BernoulliNB()
l = LogisticRegression()
d = DecisionTreeClassifier()
r = RandomForestClassifier()
gb= GradientBoostingClassifier()
kn= KNeighborsClassifier()
ab= AdaBoostClassifier()
mn= MultinomialNB()

def algo_test(x, y):
    modeller=[ b, l, d, r, gb, kn, ab, mn]
    isimler=["BernoulliNB", "LogisticRegression", "DecisionTreeClassifier", 
             "RandomForestClassifier", "GradientBoostingClassifier", "KNeighborsClassifier",
             "AdaBoostClassifier", "MultinomialNB"]

    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=.20, random_state = 42)
    
    accuracy = []
    precision = []
    recall = []
    f1 = []
    mdl=[]

    print("Veriler hazır modeller deneniyor")
    for model in modeller:
        print(model, " modeli eğitiliyor!..")
        model=model.fit(x_train,y_train)
        tahmin=model.predict(x_test)
        mdl.append(model)
        accuracy.append(accuracy_score(y_test, tahmin))
        precision.append(precision_score(y_test, tahmin, average="micro"))
        recall.append(recall_score(y_test, tahmin, average="micro"))
        f1.append(f1_score(y_test, tahmin, average="micro"))
        print(confusion_matrix(y_test, tahmin))

    print("Eğitim tamamlandı.")
    
    metrics=pd.DataFrame(columns=["Accuracy", "Precision", "Recall", "F1", "Model"], index=isimler)
    metrics["Accuracy"] = accuracy
    metrics["Precision"] = precision  
    metrics["Recall"] = recall
    metrics["F1"] = f1
    metrics["Model"]=mdl

    metrics.sort_values("F1", ascending=False, inplace=True)

    print("En başarılı model: ", metrics.iloc[0].name)
    model=metrics.iloc[0,-1]
    tahmin=model.predict(np.array(x_test) if model==kn else x_test)
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, tahmin))
    print("classification Report:")
    print(classification_report(y_test, tahmin))
    print("Diğer Modeller:")
    
    return metrics.drop("Model", axis=1)

In [49]:
algo_test(yenix,y)

Veriler hazır modeller deneniyor
BernoulliNB()  modeli eğitiliyor!..
[[556  72]
 [153 486]]
LogisticRegression()  modeli eğitiliyor!..
[[570  58]
 [ 54 585]]
DecisionTreeClassifier()  modeli eğitiliyor!..
[[497 131]
 [112 527]]
RandomForestClassifier()  modeli eğitiliyor!..
[[559  69]
 [ 61 578]]
GradientBoostingClassifier()  modeli eğitiliyor!..
[[575  53]
 [ 78 561]]
KNeighborsClassifier()  modeli eğitiliyor!..
[[512 116]
 [128 511]]
AdaBoostClassifier()  modeli eğitiliyor!..
[[552  76]
 [119 520]]
MultinomialNB()  modeli eğitiliyor!..
[[562  66]
 [ 67 572]]
Eğitim tamamlandı.
En başarılı model:  LogisticRegression
Confusion Matrix:
[[570  58]
 [ 54 585]]
classification Report:
              precision    recall  f1-score   support

           0       0.91      0.91      0.91       628
           1       0.91      0.92      0.91       639

    accuracy                           0.91      1267
   macro avg       0.91      0.91      0.91      1267
weighted avg       0.91      0.91      

,Accuracy,Precision,Recall,F1
LogisticRegression,0.911602,0.911602,0.911602,0.911602
RandomForestClassifier,0.897395,0.897395,0.897395,0.897395
GradientBoostingClassifier,0.896606,0.896606,0.896606,0.896606
MultinomialNB,0.895028,0.895028,0.895028,0.895028
AdaBoostClassifier,0.846093,0.846093,0.846093,0.846093
BernoulliNB,0.822415,0.822415,0.822415,0.822415
DecisionTreeClassifier,0.808208,0.808208,0.808208,0.808208
KNeighborsClassifier,0.807419,0.807419,0.807419,0.807419


Makine öğrenmesi algoritmaları ile elde edilen sonuçlara göre en yüksek performans RandomForestClassifier ve Logistic Regression modeli tarafından sağlanmış ve %91 doğruluk accuracy, precision, recall ve F1 skorlarına ulaşılmıştır. 
Derin öğrenme modeli de aynı şekilde %92 doğruluk oranı ve 0.2525 loss değeri ile tüm makine öğrenmesi modellerine kıyasla daha yüksek bir performans sergilemiştir.
Support Vector Machine (SVM) algoritması ile model %87 doğruluk oranı yakalmıştır.

According to the results obtained with machine learning algorithms, the highest performance was provided by the RandomForestClassifier and Logistic Regression model, achieving 91% accuracy, precision, recall, and F1 scores.

The deep learning model also exhibited higher performance compared to all machine learning models, with a 92% accuracy rate and a loss value of 0.2525.
The Support Vector Machine (SVM) algorithm achieved an 87% accuracy rate.